In [2]:
import pandas as pd
import ir_datasets

#### Usando a chave exata registrada no ir_datasets

In [3]:
dataset_dl23 = ir_datasets.load('msmarco-passage-v2/trec-dl-2023/judged')

#### A. Extrair os Qrels (Julgamentos do NIST)

In [4]:
qrels_dl23 = pd.DataFrame(dataset_dl23.qrels_iter())
print(f"Total de rótulos humanos (qrels): {len(qrels_dl23)}")

Total de rótulos humanos (qrels): 22327


#### B. Extrair os textos das consultas julgadas

In [5]:
queries_dl23 = []
for q in dataset_dl23.queries_iter():
    queries_dl23.append({
        'query_id': q.query_id,
        'query_text': q.text
    })

df_dl23_queries = pd.DataFrame(queries_dl23)
print(f"Total de consultas julgadas carregadas: {len(df_dl23_queries)}")

Total de consultas julgadas carregadas: 82


#### Carregar partição de treino do MS MARCO v2 Passage

In [9]:

print("Amostrando 1.000 consultas de treino do MS MARCO v2...")

# Carrega a partição de treino de passagens do MS MARCO v2
dataset_train = ir_datasets.load('msmarco-passage-v2/train')

train_queries = []
target_sample = 1000

# Iterador direto no fluxo de dados (sem carregar tudo no disco)
for query in dataset_train.queries_iter():
    train_queries.append({
        'query_id': query.query_id,
        'query_text': query.text
    })
    if len(train_queries) >= target_sample:
        break

# Cria a variável df_train_queries na memória
df_train_queries = pd.DataFrame(train_queries)

print(f"Sucesso! Variável df_train_queries criada na RAM com {len(df_train_queries)} consultas.")

Amostrando 1.000 consultas de treino do MS MARCO v2...
Sucesso! Variável df_train_queries criada na RAM com 1000 consultas.


In [26]:
import shutil
from pathlib import Path
import pandas as pd
import ir_datasets
from tqdm import tqdm

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)
ir_cache_file = Path.home() / ".ir_datasets" / "downloads" / "05946bac48a8ffee62e160213eab3fda"

print("==================================================")
print("ETAPA 1 & 2: RECONSTRUINDO PARES (TREINO E AVALIAÇÃO)")
print("==================================================")

ds_train = ir_datasets.load("msmarco-passage-v2/train")
ds_trec = ir_datasets.load("msmarco-passage-v2/trec-dl-2023")

# Treino (1.000 queries)
queries_train = {}
for q in ds_train.queries_iter():
    queries_train[str(q.query_id)] = q.text
    if len(queries_train) >= 1000: break

target_qids = set(queries_train.keys())
train_candidates = [
    {"query_id": str(q.query_id), "doc_id": q.doc_id, "relevance_label": q.relevance, "query_text": queries_train[str(q.query_id)]}
    for q in ds_train.qrels_iter() if str(q.query_id) in target_qids
]
df_train = pd.DataFrame(train_candidates)

# TREC DL 2023
queries_trec = {str(q.query_id): q.text for q in ds_trec.queries_iter()}
trec_qids = set(queries_trec.keys())
trec_candidates = [
    {"query_id": str(q.query_id), "doc_id": q.doc_id, "relevance_label": q.relevance, "query_text": queries_trec[str(q.query_id)]}
    for q in ds_trec.qrels_iter() if str(q.query_id) in trec_qids
]
df_trec = pd.DataFrame(trec_candidates)

all_doc_ids = list(set(df_train["doc_id"].tolist() + df_trec["doc_id"].tolist()))

print("==================================================")
print("ETAPA 3: LENDO O ARQUIVO DE 25 GB LOCAL")
print("==================================================")

docstore = ds_train.docs_store()
docs_map = {}

# Como o arquivo já existe, isso aqui vai rodar voando!
for doc in tqdm(docstore.get_many_iter(all_doc_ids), total=len(all_doc_ids), desc="Extraindo textos"):
    docs_map[doc.doc_id] = doc.text

df_train["passage_text"] = df_train["doc_id"].map(docs_map)
df_trec["passage_text"] = df_trec["doc_id"].map(docs_map)

path_train_csv = output_dir / "msmarco_v2_train_candidates_with_text.csv"
path_trec_csv = output_dir / "trec_dl_2023_candidates_with_text.csv"

df_train.to_csv(path_train_csv, index=False)
df_trec.to_csv(path_trec_csv, index=False)

print(f"\n[SUCESSO] Treino salvo em: {path_train_csv}")
print(f"[SUCESSO] TREC salvo em: {path_trec_csv}")

print("\n==================================================")
print("ETAPA 4: DESTRUINDO O ARQUIVO GIGANTE")
print("==================================================")

if ir_cache_file.exists():
    ir_cache_file.unlink()
    print("[OK] Arquivo de 25 GB deletado! Espaço em disco recuperado.")
else:
    print("O arquivo já não estava lá.")

ETAPA 1 & 2: RECONSTRUINDO PARES (TREINO E AVALIAÇÃO)
ETAPA 3: LENDO O ARQUIVO DE 25 GB LOCAL


Extraindo textos:   0%|          | 0/22883 [00:00<?, ?it/s]
[INFO] [finished] [04:58] [70file] [ 4.26s/file]                                
[04:58] [70file] [ 4.26s/file]   | 0/22883 [04:58<?, ?it/s]
Extraindo textos: 100%|██████████| 22883/22883 [05:00<00:00, 76.10it/s]  



[SUCESSO] Treino salvo em: data/msmarco_v2_train_candidates_with_text.csv
[SUCESSO] TREC salvo em: data/trec_dl_2023_candidates_with_text.csv

ETAPA 4: DESTRUINDO O ARQUIVO GIGANTE
[OK] Arquivo de 25 GB deletado! Espaço em disco recuperado.
